# Import data
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [ ]:
import os
import dotenv
from langchain_postgres import PGEngine
from pydantic import SecretStr

dotenv.load_dotenv()
ENV_PG_CONNECTION_STRING = os.getenv("ENV_PG_CONNECTION_STRING")
ENV_ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# Obsidian Vault

In [ ]:
from sqlalchemy import create_engine
from agent_assistant.utils.chunker.text import TextChunker
from agent_assistant.retriever.obsidian import ObsidianChunkStore, ObsidianDocumentStore

# エンジン初期化
assert ENV_PG_CONNECTION_STRING is not None
assert ENV_ENV_GEMINI_API_KEY is not None

sa_engine = create_engine(ENV_PG_CONNECTION_STRING)
pg_engine = PGEngine.from_connection_string(ENV_PG_CONNECTION_STRING, pool_size=5)
chunk_store = ObsidianChunkStore(pg_engine, SecretStr(ENV_ENV_GEMINI_API_KEY))
obsidian_store = ObsidianDocumentStore(
    "obsidian_vault", chunk_store, sa_engine, TextChunker(chunk_size=300, chunk_overlap=30)
)

In [ ]:
from pathlib import Path
from agent_assistant.loader.obsidian import VaultLoader

vault_path = Path("../docs/dataset_obsidian/")
loader = VaultLoader(vault_path)
docs = loader.load()

print(f"{len(docs)} 件のノートを読み込みました")
print(docs[0])

In [ ]:
obsidian_store.connect()
# obsidian_store.import_documents(docs)

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from agent_assistant.utils.chunker.text import TextChunker

note_path = Path("../docs/dataset_obsidian/").resolve() / "02_Daily/2025-12-28.md"
TextChunker().chunk([Document(note_path.read_text())])

In [ ]:
assert obsidian_store._chunk_vs is not None
obsidian_store._chunk_vs.similarity_search_with_relevance_scores("プロンプトエンジニアリング", k=5)

In [ ]:
from IPython.display import display, Markdown
from agent_assistant.connector import format_documents

results = obsidian_store.search_documents(
    "コンピュータサイエンス",
    search_type="similarity_score_threshold",
    k=3,
    score_threshold=0.65,
)
results = format_documents(results, obsidian_store)
display(Markdown(results))